In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Is_solvated)=
# Is solvated

*Checking a water-density solvation criterion per structure.*

The function {func}`molsysmt.build.is_solvated` checks whether indexed waters have
a global number density above **15 molecules/nm³** in each selected periodic box.
One selected structure returns a Python `bool`; multiple structures return a
one-dimensional Boolean NumPy array. Equality does not pass. Equivalent units
are converted explicitly, with an eight-machine-epsilon relative comparison
tolerance around the boundary to account for floating-point roundoff.

This heuristic does not establish solvent coverage, containment, equilibration,
chemical readiness or ionic strength. Ions and other solvents alone do not satisfy
its water prerequisite. The predicate reads the indexed water inventory and box
metadata; it does not require atomic coordinates. Native inputs do not materialize
coordinates; file and foreign adapters retain their own loading costs. Reuse a
resident native system for repeated queries when those costs matter.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.build.is_solvated`.
:::


## Basic usage

Let's show how this function works using Met-enkephalin:


In [2]:
import molsysmt as msm


In [3]:
molsys = msm.convert(msm.systems['Met-enkephalin']['met_enkephalin.h5msm'], to_form='molsysmt.MolSys')


:::{note}
:class: dropdown

**Demo Systems Catalog**

This tutorial uses demonstration datasets provided by MolSysMT. To explore the full catalog of bundled systems, forms, and file paths, visit the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` guide.
:::


First, let's complete terminal cappings and hydrogens for the peptide:


In [4]:
molsys = msm.build.add_missing_terminal_cappings(molsys)
molsys = msm.build.add_missing_hydrogens(molsys)


We check if the system is solvated using {func}`molsysmt.build.is_solvated`:


In [5]:
msm.build.is_solvated(molsys)


False

Now we solvate the system in a cubic water box using {func}`molsysmt.build.solvate`:


In [6]:
molsys = msm.build.solvate(molsys, box_shape='cubic', clearance='14.0 angstroms', water_model='TIP3P')


We check again whether the resulting global water density passes the predicate:


In [7]:
msm.build.is_solvated(molsys)


True

In [8]:
msm.get(molsys, n_waters=True)


1389

## Checking an ensemble

The following ensemble repeats the prepared structure three times. We then enlarge
only the second box to demonstrate a mixed density result. These are synthetic
box changes with unchanged coordinates, not an equilibrated or revalidated solvent
model. The independent extraction leaves the earlier source unchanged.


In [9]:
molsys = msm.extract(molsys, structure_indices=[0, 0, 0])
puw = msm.pyunitwizard
box = puw.get_value(msm.get(molsys, box=True), to_unit='nm').copy()
box[1] *= 2.0
msm.set(molsys, box=puw.quantity(box, 'nm'))
result = msm.build.is_solvated(molsys)
result.tolist()

[True, False, True]

## Selecting structures

`structure_indices` addresses source positions, independently of structure-ID
labels. Order and repeated indices are retained. The addon follows this same
selection; it does not substitute the visible structure.


In [10]:
msm.build.is_solvated(molsys, structure_indices=[2, 1, 2, 0]).tolist()

[True, False, True, True]

In [11]:
msm.build.is_solvated(molsys, structure_indices=1)

False

## Interpreting the result

There is no implicit first/any/all aggregation. Choose an ensemble interpretation
explicitly when the result is an array:


In [12]:
bool(result.all()), bool(result.any())

(False, True)

An empty selection returns a Boolean array of shape `(0,)`. Check its size before
reducing: NumPy's `all()` on an empty array is true by convention, not evidence
that any structure passed. One selected structure always yields a Python bool.


In [13]:
msm.build.is_solvated(molsys, structure_indices=[]).shape

(0,)

## Handling missing or unusable data

No indexed waters or no box yields false for every selected structure. A missing
water inventory is different: coordinate-only input cannot establish the water
count and raises an explicit unavailable-attribute diagnostic. Nonpositive or
nonfinite available box volumes are refused. Negative, malformed and out-of-range
structure indices are rejected, even when no waters are present; `None` is not a
structure selection. A stable inventory with no structure axis or box returns
false. No coordinates, chemical state, interaction analysis or scene is changed.


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Convert <Tutorial_Convert>`: Convert molecular systems between different forms with {func}`molsysmt.basic.convert`.
- {ref}`Add missing terminal cappings <Tutorial_Add_missing_terminal_cappings>`: Add terminal cappings to peptides or proteins with {func}`molsysmt.build.add_missing_terminal_cappings`.
- {ref}`Add missing hydrogens <Tutorial_Add_missing_hydrogens>`: Add missing hydrogen atoms to a molecular system with {func}`molsysmt.build.add_missing_hydrogens`.
- {ref}`Is solvated <Tutorial_Is_solvated>`: Check whether a molecular system contains solvent molecules with {func}`molsysmt.build.is_solvated`.
- {ref}`Solvate <Tutorial_Solvate>`: Add solvent molecules around a molecular system with {func}`molsysmt.build.solvate`.
- {ref}`Get <Tutorial_Get>`: Retrieve attribute values from a molecular system with {func}`molsysmt.basic.get`.
:::
